# Playground: quick model

exp01 workflow ([card](../sections/baseline/experiments/exp01_lecture_models.md)) on tiny model, data subset, small images. Training code: `src/core/training/cv.py`, configs: `configs/`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from core.training.cv import Config, CrossValidator
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.training.config import CLASSES
from core.analysis.plots import use_style

CONFIGS = ["configs/quick.yaml"]

use_style()

## Training


In [ ]:
results = {}
for path in CONFIGS:
    result = CrossValidator(Config.from_yaml(path)).run()
    results[result.config.name] = result
SERIES = dict(zip(results, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
fold_ap_df = pd.concat(
    {name: r.fold_ap for name, r in results.items()}, names=["model"]
)

## Training curves


In [ ]:
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over folds", xlabel="epoch")
axes[1].legend()
plt.show()

## Precision-recall curves


In [ ]:
best = {name: r.scores.best_f1() for name, r in results.items()}
labels = next(iter(results.values())).labels
fig, axes = plt.subplots(1, len(CLASSES), figsize=(3.3 * len(CLASSES), 3.6), sharey=True)
for c, (cls, ax) in enumerate(zip(CLASSES, axes)):
    y = labels[:, c]
    ax.axhline(y.mean(), color="#52514e", ls="--", lw=0.8)
    for name, r in results.items():
        precision, recall, _ = precision_recall_curve(y, r.probs[:, c])
        b = best[name].loc[cls]
        ap = average_precision_score(y, r.probs[:, c])
        ax.step(recall, precision, where="post", color=SERIES[name], label=f"AP {ap:.2f} · F1 {b.f1:.2f} @ t {b.threshold:.2f}")
        ax.plot(b.recall, b.precision, "o", ms=7, color=SERIES[name], mec="white", mew=1.5)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="recall", title=f"{cls}  (n+ {y.sum():.0f}, prevalence {y.mean():.2f})")
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.2), handlelength=1.2)
axes[0].set_ylabel("precision")
models = [plt.Line2D([], [], color=SERIES[name]) for name in results]
fig.legend(models, [f"{name}  macro F1 {best[name].f1.mean():.3f}" for name in results],
           loc="upper left", ncol=len(results), bbox_to_anchor=(0, 1.1))
fig.tight_layout()
plt.show()

In [ ]:
f1_df = pd.DataFrame({name: b.f1 for name, b in best.items()}).T
f1_df.assign(macro_F1=f1_df.mean(axis=1)).sort_values("macro_F1", ascending=False).round(3)

## cAP comparison


In [ ]:
fold_ap_df.groupby("model", sort=False).mean().sort_values(
    "mAP", ascending=False
).round(3)

In [ ]:
fold_ap_df.round(3)